# Hybrid Search Demo

Combine dense (semantic) and sparse (BM25) retrieval using Reciprocal Rank Fusion.

**Why Hybrid?**
- Dense: Good at semantic similarity, misses exact terms
- Sparse: Good at exact matches, misses synonyms
- Hybrid: Best of both worlds

**Also covers:**
- Multi-query expansion
- HyDE (Hypothetical Document Embeddings)

**Prerequisites:**
```bash
pip install rank-bm25 numpy
ollama pull nomic-embed-text
ollama pull qwen3:4b  # For query expansion/HyDE
```

In [1]:
# Setup
import subprocess
import requests
import numpy as np
from rank_bm25 import BM25Okapi

def check_setup():
    try:
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            print("✓ Ollama running")
            if "nomic-embed-text" in result.stdout:
                print("✓ nomic-embed-text available")
            if "qwen3" in result.stdout:
                print("✓ qwen3 available (for HyDE)")
        return True
    except Exception as e:
        print(f"✗ Setup issue: {e}")
        return False

check_setup()

# Sample document corpus
DOCUMENTS = [
    {"id": "doc1", "content": "Remote work policy allows employees to work from home up to 3 days per week. Manager approval required."},
    {"id": "doc2", "content": "Error code TS-7492 indicates a database connection timeout. Check your connection string and firewall settings."},
    {"id": "doc3", "content": "Annual leave entitlement is 25 days for full-time employees. Unused days can be carried over to Q1."},
    {"id": "doc4", "content": "The WFH guidelines were updated in January 2024 to include hybrid work arrangements."},
    {"id": "doc5", "content": "Performance reviews occur quarterly. Employees must complete self-assessment 2 weeks before review date."},
    {"id": "doc6", "content": "IT support hotline: ext 4567. For urgent issues outside business hours, use the emergency pager."},
    {"id": "doc7", "content": "Vacation policy: Employees accrue 2.08 days per month. Maximum carry-over is 5 days."},
    {"id": "doc8", "content": "The telecommuting program requires completion of the home office safety checklist."},
]

print(f"Corpus: {len(DOCUMENTS)} documents")

✓ Ollama running
✓ nomic-embed-text available
✓ qwen3 available (for HyDE)
Corpus: 8 documents


---

## 1. Dense Retrieval (Embeddings)

In [2]:
def get_embedding(text: str, model: str = "nomic-embed-text") -> list[float]:
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": model, "prompt": text}
    )
    return response.json()["embedding"]

def cosine_similarity(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Pre-compute document embeddings
print("Computing document embeddings...")
doc_embeddings = {doc["id"]: get_embedding(doc["content"]) for doc in DOCUMENTS}
print(f"✓ {len(doc_embeddings)} embeddings computed")

def dense_search(query: str, top_k: int = 5) -> list[tuple[str, float]]:
    """Search using embedding similarity."""
    query_emb = get_embedding(query)
    
    scores = []
    for doc in DOCUMENTS:
        sim = cosine_similarity(query_emb, doc_embeddings[doc["id"]])
        scores.append((doc["id"], sim))
    
    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

# Test dense search
query = "WFH rules"
results = dense_search(query)
print(f"\nDense search for: \"{query}\"")
for doc_id, score in results[:3]:
    doc = next(d for d in DOCUMENTS if d["id"] == doc_id)
    print(f"  {doc_id} ({score:.3f}): {doc['content'][:60]}...")

Computing document embeddings...
✓ 8 embeddings computed

Dense search for: "WFH rules"
  doc4 (0.581): The WFH guidelines were updated in January 2024 to include h...
  doc7 (0.550): Vacation policy: Employees accrue 2.08 days per month. Maxim...
  doc1 (0.537): Remote work policy allows employees to work from home up to ...


---

## 2. Sparse Retrieval (BM25)

In [3]:
# Build BM25 index
tokenized_docs = [doc["content"].lower().split() for doc in DOCUMENTS]
bm25 = BM25Okapi(tokenized_docs)

def sparse_search(query: str, top_k: int = 5) -> list[tuple[str, float]]:
    """Search using BM25."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    
    results = [(DOCUMENTS[i]["id"], scores[i]) for i in range(len(DOCUMENTS))]
    return sorted(results, key=lambda x: x[1], reverse=True)[:top_k]

# Test sparse search
query = "TS-7492"  # Exact identifier - BM25 should excel
results = sparse_search(query)
print(f"Sparse search for: \"{query}\"")
for doc_id, score in results[:3]:
    doc = next(d for d in DOCUMENTS if d["id"] == doc_id)
    print(f"  {doc_id} ({score:.3f}): {doc['content'][:60]}...")

Sparse search for: "TS-7492"
  doc2 (1.579): Error code TS-7492 indicates a database connection timeout. ...
  doc1 (0.000): Remote work policy allows employees to work from home up to ...
  doc3 (0.000): Annual leave entitlement is 25 days for full-time employees....


---

## 3. Reciprocal Rank Fusion (RRF)

Merge dense and sparse results using rank-based fusion.

In [4]:
def reciprocal_rank_fusion(
    result_lists: list[list[tuple[str, float]]], 
    k: int = 60
) -> list[tuple[str, float]]:
    """
    Merge ranked lists using RRF.
    
    Formula: RRF(d) = Σ 1/(k + rank)
    k=60 is the standard default from the original paper.
    """
    rrf_scores = {}
    
    for results in result_lists:
        for rank, (doc_id, _) in enumerate(results):
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0.0
            rrf_scores[doc_id] += 1.0 / (k + rank)
    
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

def hybrid_search(query: str, top_k: int = 5) -> list[tuple[str, float]]:
    """Combine dense and sparse search with RRF."""
    dense_results = dense_search(query, top_k=10)
    sparse_results = sparse_search(query, top_k=10)
    
    return reciprocal_rank_fusion([dense_results, sparse_results])[:top_k]

# Compare all three approaches
test_queries = [
    "WFH policy",           # Conceptual - dense should help
    "error TS-7492",        # Identifier - sparse should help
    "vacation carry over",  # Mixed - hybrid should excel
]

print("Search Comparison")
print("=" * 70)

for query in test_queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 50)
    
    dense_results = dense_search(query, top_k=3)
    sparse_results = sparse_search(query, top_k=3)
    hybrid_results = hybrid_search(query, top_k=3)
    
    print(f"  Dense:  {[r[0] for r in dense_results]}")
    print(f"  Sparse: {[r[0] for r in sparse_results]}")
    print(f"  Hybrid: {[r[0] for r in hybrid_results]}")

Search Comparison

Query: "WFH policy"
--------------------------------------------------
  Dense:  ['doc7', 'doc1', 'doc4']
  Sparse: ['doc4', 'doc1', 'doc2']
  Hybrid: ['doc4', 'doc1', 'doc7']

Query: "error TS-7492"
--------------------------------------------------
  Dense:  ['doc2', 'doc6', 'doc8']
  Sparse: ['doc2', 'doc1', 'doc3']
  Hybrid: ['doc2', 'doc6', 'doc4']

Query: "vacation carry over"
--------------------------------------------------
  Dense:  ['doc7', 'doc3', 'doc1']
  Sparse: ['doc7', 'doc3', 'doc1']
  Hybrid: ['doc7', 'doc3', 'doc1']


---

## 4. Multi-Query Expansion

Generate query variations to improve recall.

In [5]:
def generate_queries(original_query: str, model: str = "qwen3:4b") -> list[str]:
    """Generate alternative queries using LLM."""
    prompt = f"""Generate 3 alternative search queries for: "{original_query}"
Each query should use different words but search for the same information.
Return only the queries, one per line. No thinking, no explanation."""
    
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    
    result = response.json().get("response", "")
    # Parse queries from response
    queries = [q.strip().strip('"').strip("'") for q in result.strip().split('\n') if q.strip()]
    queries = [q for q in queries if len(q) > 5 and len(q) < 100][:3]
    
    return [original_query] + queries

def multi_query_search(query: str, top_k: int = 5) -> list[tuple[str, float]]:
    """Search with multiple query variations and merge results."""
    queries = generate_queries(query)
    print(f"  Generated queries: {queries}")
    
    all_results = []
    for q in queries:
        results = hybrid_search(q, top_k=5)
        all_results.append(results)
    
    return reciprocal_rank_fusion(all_results)[:top_k]

# Test multi-query
try:
    query = "how to work from home"
    print(f"\nMulti-Query Search: \"{query}\"")
    print("-" * 50)
    results = multi_query_search(query)
    print(f"  Final results: {[r[0] for r in results[:3]]}")
except Exception as e:
    print(f"Multi-query failed: {e}")


Multi-Query Search: "how to work from home"
--------------------------------------------------
  Generated queries: ['how to work from home', 'steps for working remotely', 'guide to home office work', 'best practices for telecommuting']
  Final results: ['doc1', 'doc8', 'doc4']


---

## 5. HyDE (Hypothetical Document Embeddings)

Generate a hypothetical answer, then search with its embedding.

In [6]:
def generate_hypothetical_answer(query: str, model: str = "qwen3:4b") -> str:
    """Generate what an answer to this query might look like."""
    prompt = f"""Write a brief passage (2-3 sentences) that would answer this question.
Base it on general knowledge of how organizations typically handle this.
No thinking, just the answer passage.

Question: {query}

Answer:"""
    
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    
    return response.json().get("response", "").strip()

def hyde_search(query: str, top_k: int = 5) -> list[tuple[str, float]]:
    """Search using hypothetical document embedding."""
    # Generate hypothetical answer
    hypothetical = generate_hypothetical_answer(query)
    print(f"  Hypothetical: {hypothetical[:100]}...")
    
    # Search with hypothetical embedding
    hypo_emb = get_embedding(hypothetical)
    
    scores = []
    for doc in DOCUMENTS:
        sim = cosine_similarity(hypo_emb, doc_embeddings[doc["id"]])
        scores.append((doc["id"], sim))
    
    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

# Test HyDE
try:
    query = "What are the rules for taking time off?"
    print(f"\nHyDE Search: \"{query}\"")
    print("-" * 50)
    
    # Compare direct vs HyDE
    direct_results = dense_search(query, top_k=3)
    hyde_results = hyde_search(query, top_k=3)
    
    print(f"  Direct dense: {[r[0] for r in direct_results]}")
    print(f"  HyDE:         {[r[0] for r in hyde_results]}")
except Exception as e:
    print(f"HyDE failed: {e}")


HyDE Search: "What are the rules for taking time off?"
--------------------------------------------------
  Hypothetical: Organizations typically establish clear policies that define eligibility, duration, and requirements...
  Direct dense: ['doc3', 'doc1', 'doc7']
  HyDE:         ['doc1', 'doc3', 'doc7']


---

## Summary

| Technique | Best For | Latency | When to Use |
|-----------|----------|---------|-------------|
| Dense only | Conceptual queries | Fast | Prototyping |
| Sparse only | Exact identifiers | Fast | Error codes, SKUs |
| Hybrid (RRF) | Mixed queries | +100ms | Production default |
| Multi-query | Ambiguous queries | +500ms | Low volume, high recall |
| HyDE | Q&A over docs | +500ms | Large vocab gap |

**Recommendation:** Start with hybrid search (dense + BM25 + RRF).
Add multi-query or HyDE only for specific query types that need it.